#### 连接MySQL数据库

In [1]:
import os
import sys

# 设置环境变量
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'

In [2]:
os.environ['JAVA_HOME'] = 'C:\Program Files\JDK1.8'

<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:1: SyntaxWarning: invalid escape sequence '\P'
C:\Users\MouKexin\AppData\Local\Temp\ipykernel_78388\4280918141.py:1: SyntaxWarning: invalid escape sequence '\P'
  os.environ['JAVA_HOME'] = 'C:\Program Files\JDK1.8'


In [3]:
from pyspark.sql import SparkSession

# 创建本地模式的 SparkSession（不依赖 Hadoop）
spark = SparkSession.builder \
    .appName("LocalSpark") \
    .master("local[*]") \
    .config("spark.jars", "D:/mysql-connector-java-5.1.7-bin.jar") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.warehouse.dir", r"file:///D:/code/python/BigDataDevelopment/myCode/spark-warehouse") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

RuntimeError: Java gateway process exited before sending its port number

In [55]:
# 推荐首先尝试这个
spark.stop()

# 使用 Maven 坐标（自动下载驱动）
spark = SparkSession.builder \
    .appName("MySQLAutoDownload") \
    .master("local[*]") \
    .config("spark.jars", "D:/mysql-connector-java-5.1.7-bin.jar") \
    .getOrCreate()

print("✅ 使用 Maven 坐标创建 SparkSession 完成")

✅ 使用 Maven 坐标创建 SparkSession 完成


In [56]:
# 快速测试 - 修改下面的配置后运行
def quick_test():
    # 修改为你的 MySQL 信息
    jdbc_url = "jdbc:mysql://localhost:3306/book_test"
    properties = {
        "user": "root",
        "password": "050221",  # 你的密码
        "driver": "com.mysql.jdbc.Driver"
    }

    try:
        df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("dbtable", "(SELECT 1 as test) as tmp") \
            .option("user", properties["user"]) \
            .option("password", properties["password"]) \
            .option("driver", properties["driver"]) \
            .load()

        print("✅ 连接成功！")
        df.show()
        return True
    except Exception as e:
        print(f"❌ 连接失败: {e}")
        return False

quick_test()

❌ 连接失败: An error occurred while calling o252.load.
: java.lang.ClassNotFoundException: com.mysql.jdbc.Driver
	at java.net.URLClassLoader.findClass(Unknown Source)
	at java.lang.ClassLoader.loadClass(Unknown Source)
	at java.lang.ClassLoader.loadClass(Unknown Source)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:46)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:103)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:41)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcRelationProvider.createRelation(JdbcRelationProvider.scala:34)
	at org.apache.spark.sql.execution.datasources.DataSou

False

In [57]:
# 停止当前会话
spark.stop()

# 使用 Maven 坐标自动下载正确的驱动
spark = SparkSession.builder \
    .appName("MySQLMaven") \
    .master("local[*]") \
    .config("spark.jars.packages", "mysql:mysql-connector-java:8.0.33") \
    .getOrCreate()

print("✅ 使用 Maven 坐标创建 SparkSession 完成")

✅ 使用 Maven 坐标创建 SparkSession 完成


In [50]:
# 4. 测试 MySQL 连接
def test_mysql_final():
    try:
        jdbc_url = "jdbc:mysql://localhost:3306/book_test"

        # 方法 1: 使用 MySQL 8.x 驱动类
        df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("dbtable", "(SELECT 1 as test_result, '成功' as message) as tmp") \
            .option("user", "root") \
            .option("password", "050221") \
            .option("driver", "com.mysql.cj.jdbc.Driver") \
            .load()

        print("✅ MySQL 8.x 驱动连接成功！")
        df.show()
        return True

    except Exception as e:
        print(f"MySQL 8.x 驱动失败: {e}")

# 运行测试
test_mysql_final()

MySQL 8.x 驱动失败: An error occurred while calling o214.load.
: java.lang.ClassNotFoundException: com.mysql.cj.jdbc.Driver
	at java.net.URLClassLoader.findClass(Unknown Source)
	at java.lang.ClassLoader.loadClass(Unknown Source)
	at java.lang.ClassLoader.loadClass(Unknown Source)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:46)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:103)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:41)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcRelationProvider.createRelation(JdbcRelationProvider.scala:34)
	at org.apache.spark.sql.execution.datasour